# Обучение SegFormer для задачи сегментации повреждений фасада

In [ ]:
!pip install -q transformers evaluate datasets albumentations accelerate

In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import evaluate
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor, TrainingArguments, Trainer
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем устройство: {device}")

## 1. Настройка путей и классов

In [ ]:
# ПУТИ В KAGGLE
DATASET_PATH = '/kaggle/input/datasets/neuromant/facade-damage-seg-301-imgs/unified_dataset'  
IMAGES_DIR = os.path.join(DATASET_PATH, 'Images')          # Папка с исходными .jpg
MASKS_DIR = os.path.join(DATASET_PATH, 'Masks')            # Папка с экспортированными из CVAT масками (RGB)

# Маппинг классов
CLASS_MAPPING = {
    0: "background",
    1: "coating_deterioration",
    2: "cracks",
    3: "masonry_degradation",
    4: "moisture_bio_damage",
    5: "vandalism",
    255: "Ignored" 
}
id2label = CLASS_MAPPING
label2id = {v: k for k, v in id2label.items()}

cvat_colors_rgb = {
    "background": [0, 0, 0],
    "coating_deterioration": [54, 21, 217],
    "cracks": [250, 50, 183],
    "masonry_degradation": [42, 125, 209],
    "moisture_bio_damage": [38, 221, 98],
    "vandalism": [222, 28, 123],
    "Ignored": [221, 255, 51]
}

## 2. Подготовка Данных (Конвертация RGB-масок в Индексные)

In [ ]:
# Чтобы не конвертировать маски каждый раз с нуля, мы сделаем это один раз и сохраним в рабочую директорию Kaggle (/kaggle/working)
PROCESSED_MASKS_DIR = '/kaggle/working/masks_indexed'
os.makedirs(PROCESSED_MASKS_DIR, exist_ok=True)

def rgb_mask_to_index_mask(rgb_mask_bgr, color_map):
    index_mask = np.zeros(rgb_mask_bgr.shape[:2], dtype=np.uint8)
    class_to_idx = {name: idx for idx, name in CLASS_MAPPING.items()} 
    
    for class_name, class_idx in class_to_idx.items():
        rgb = color_map[class_name]
        bgr = [rgb[2], rgb[1], rgb[0]]
        
        lower = np.array(bgr, dtype=np.uint8)
        upper = np.array(bgr, dtype=np.uint8)
        class_pixels = cv2.inRange(rgb_mask_bgr, lower, upper)
        index_mask[class_pixels == 255] = class_idx
        
    return index_mask

# Собираем пары (image, mask)
from glob import glob
mask_files = glob(os.path.join(MASKS_DIR, "*.png"))

dataset_pairs = []

print("Конвертация масок и сбор пар...")
for mask_path in tqdm(mask_files, total=len(mask_files)):
    filename = os.path.basename(mask_path)
    
    # Ищем исходную картинку
    image_path_png = os.path.join(IMAGES_DIR, filename)
    image_path_jpg = os.path.join(IMAGES_DIR, os.path.splitext(filename)[0] + '.jpg')
    
    img_path = None
    if os.path.exists(image_path_png):
        img_path = image_path_png
    elif os.path.exists(image_path_jpg):
        img_path = image_path_jpg
        
    if img_path is None:
        continue
        
    # Читаем RGB маску и конвертируем
    rgb_mask_bgr = cv2.imread(mask_path, cv2.IMREAD_COLOR)
    if rgb_mask_bgr is None: continue
        
    index_mask = rgb_mask_to_index_mask(rgb_mask_bgr, cvat_colors_rgb)
    
    # Сохраняем
    indexed_mask_path = os.path.join(PROCESSED_MASKS_DIR, filename)
    cv2.imwrite(indexed_mask_path, index_mask)
    
    dataset_pairs.append({
        "image": img_path,
        "mask": indexed_mask_path
    })

print(f"Собрано {len(dataset_pairs)} пар картинок и масок.")

## 3. Train/Val Split

Разделим на обучение (80%) и валидацию (20%)

In [ ]:
train_pairs, val_pairs = train_test_split(dataset_pairs, test_size=0.2, random_state=42)
print(f"Обучающая выборка: {len(train_pairs)}")
print(f"Валидационная выборка: {len(val_pairs)}")

## 4. PyTorch Dataset & Augmentations

In [ ]:
class DamageSegmentationDataset(Dataset):
    def __init__(self, dataset_pairs, transform=None):
        self.dataset_pairs = dataset_pairs
        self.transform = transform
        # ImageProcessor нормирует картинки под SegFormer
        self.processor = SegformerImageProcessor.from_pretrained("nvidia/mit-b2")

    def __len__(self):
        return len(self.dataset_pairs)

    def __getitem__(self, idx):
        pair = self.dataset_pairs[idx]
        
        # Считываем изображение (RGB для Albumentations)
        image = cv2.imread(pair["image"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Считываем индексную маску
        mask = cv2.imread(pair["mask"], cv2.IMREAD_GRAYSCALE)
        
        # Применяем аугментации
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # Подготовка под HuggingFace Segformer Processor
        # Он занимается ресайзом и нормализацией (ImageNet norm)
        # HuggingFace ожидает PIL картинку
        inputs = self.processor(images=Image.fromarray(image), segmentation_maps=mask, return_tensors="pt")
        
        # Убираем лишнюю размерность batch, созданную процессором
        inputs = {k: v.squeeze() for k, v in inputs.items()}
        
        return inputs

# Аугментации на тренировку (чуть добавляем вариативности)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
])

# Валидации ессно без аугментаций
val_transform = None

train_dataset = DamageSegmentationDataset(train_pairs, transform=train_transform)
val_dataset = DamageSegmentationDataset(val_pairs, transform=val_transform)

## 5. Вычисление Весов Классов (Loss Weights)

У нас сильный дисбаланс, поэтому мы вычислим веса для CrossEntropyLoss, чтобы штрафовать редкие классы сильнее.

In [ ]:
# Веса рассчитаны скриптом на всем объединенном датасете (Median Frequency Balancing)
weights = [0.0485, 0.4808, 14.9300, 4.4474, 0.5633, 7.4649]

class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)
print("Веса классов для CrossEntropyLoss:")
for i, w in enumerate(weights):
    print(f"Класс {id2label[i]}: {w:.4f}")

## Инициализация wandb

In [ ]:
import wandb
wandb.login() # Тебе нужно будет вставить API ключ от твоего аккаунта W&B
wandb.init(project="facade-damage-segformer", name="5_classes_run1")

## 6. Инициализация Модели и Метрик

Используем кастомный класс `WeightedTrainer`, чтобы передать наши веса в функцию потерь.

In [ ]:
from torch import nn

# Кастомный Trainer с весами
class ClassWeightsTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Скейл масок к размеру логитов (1/4 от оригинального изображения)
        upsampled_logits = nn.functional.interpolate(
            logits, 
            size=labels.shape[-2:], 
            mode="bilinear", 
            align_corners=False
        )
        
        # ignore_index=255 значит, что лосс не считается на пикселях класса Ignored!
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor, ignore_index=255)
        loss = loss_fct(upsampled_logits, labels)
        
        return (loss, outputs) if return_outputs else loss


# Загружаем метрику mIoU
metric = evaluate.load("mean_iou")

def compute_metrics(eval_pred):
    with torch.no_grad():
        logits, labels = eval_pred
        logits_tensor = torch.from_numpy(logits)
        
        # Скейл логитов до оригинального размера
        logits_tensor = nn.functional.interpolate(
            logits_tensor,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False,
        ).argmax(dim=1)
        
        pred_labels = logits_tensor.detach().cpu().numpy()
        
        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=len(id2label) - 1, 
            ignore_index=255,
            reduce_labels=False,
        )
        
        # БЕЗОПАСНОЕ извлечение метрик (перевод всего в float)
        safe_metrics = {}
        
        # Основная метрика mIoU
        if "mean_iou" in metrics:
            safe_metrics["mean_iou"] = float(metrics["mean_iou"])
            
        if "mean_accuracy" in metrics:
            safe_metrics["mean_accuracy"] = float(metrics["mean_accuracy"])
            
        if "overall_accuracy" in metrics:
            safe_metrics["overall_accuracy"] = float(metrics["overall_accuracy"])
        
        # Вытаскиваем IoU по каждому классу отдельно
        if "per_category_iou" in metrics:
            # Превращаем numpy array в обычный питоновский list
            per_category_iou = metrics["per_category_iou"]
            if isinstance(per_category_iou, np.ndarray):
                per_category_iou = per_category_iou.tolist()
                
            for i, val in enumerate(per_category_iou):
                if i < len(id2label) - 1:
                    # Обязательно кастуем в float
                    safe_metrics[f"iou_{id2label[i]}"] = float(val) if not np.isnan(val) else 0.0
            
        return safe_metrics


# Загрузка модели (Берем ImageNet encoder B2)
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b2",
    num_labels=len(id2label) - 1, # 6 рабочих классов (0-5)
    id2label={k: v for k,v in id2label.items() if k != 255},
    label2id={v: k for k,v in id2label.items() if k != 255},
    ignore_mismatched_sizes=True # Важно, т.к. мы меняем голову под свои 4 класса
)

## 7. Параметры обучения

Ставим 30-50 эпох, так как датасет небольшой. С оптимизатором AdamW и Cosine расписанием, чтобы  SegFormer быстро выучился.

In [ ]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/segformer-b2-facade-damage",
    report_to="wandb",
    learning_rate=6e-5,
    num_train_epochs=50,
    per_device_train_batch_size=8, # Кажется, неплохо влезло на GPU
    per_device_eval_batch_size=8,
    save_total_limit=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    eval_steps=10,
    remove_unused_columns=False,
    push_to_hub=False,
    load_best_model_at_end=True,
    metric_for_best_model="mean_iou",
)

trainer = ClassWeightsTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

## 8. Запуск Обучения

In [ ]:
print("Погнали учиться!")
trainer.train()

## 9. Анализ результатов после повторного инференса на валидационной выборке

Загружаем ранее обученную модель

In [ ]:
MODEL_PATH = '/kaggle/input/models/neuromant/segformer-b2-facade-damage/transformers/default/1/segformer_weights' 

print("Загрузка процессора и модели...")
processor = SegformerImageProcessor.from_pretrained("nvidia/mit-b2")
model = SegformerForSemanticSegmentation.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

In [ ]:
import seaborn as sns 
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import DataLoader
from PIL import Image
import cv2 

Функция для сбора предсказаний модели

  Эта функция пройдет по валидационному набору данных и соберет все истинные маски и предсказанные моделью маски. Она
  также сохранит несколько примеров для визуализации.

In [ ]:
def collect_predictions(model, val_dataset, device, processor, num_samples_to_visualize=5):
    """
    Собирает истинные и предсказанные маски для всего валидационного набора данных,
    а также несколько примеров для визуализации.

    Параметры:
        model: Обученная модель SegFormer.
        val_dataset: Объект PyTorch Dataset для валидационных данных.
        device: Устройство, на котором будет выполняться инференс ('cuda' или 'cpu').
        processor: SegformerImageProcessor, используемый для предобработки изображений.
        num_samples_to_visualize: Количество случайных примеров для сохранения для визуализации.

    Возвращает:
        all_true_masks_flat: Одномерный массив всех истинных пиксельных меток.
        all_pred_masks_flat: Одномерный массив всех предсказанных пиксельных меток.
        sample_images: Список оригинальных изображений для визуализации.
        sample_true_masks: Список истинных масок для визуализации.
        sample_pred_masks: Список предсказанных масок для визуализации.
    """
    model.eval() # Переводим модель в режим оценки
    model.to(device)

    val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False) # Используем batch_size=1 для простоты

    all_true_masks_flat = []
    all_pred_masks_flat = []

    sample_images = []
    sample_true_masks = []
    sample_pred_masks = []

    # Для отслеживания, сколько примеров для визуализации уже собрано
    visualized_count = 0

    print("Собираем предсказания для валидационного набора данных...")
    with torch.no_grad(): 
        for i, batch in tqdm(enumerate(val_dataloader), total=len(val_dataloader)):
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits

            # Изменяем размер логитов до размера оригинальной маски
            upsampled_logits = torch.nn.functional.interpolate(
                logits,
                size=labels.shape[-2:], # Размер истинной маски
                mode="bilinear",
                align_corners=False,
            )

            # Получаем предсказанные классы
            pred_labels = upsampled_logits.argmax(dim=1)

            # Перемещаем на CPU и конвертируем в numpy для дальнейшей обработки
            true_mask_np = labels.cpu().numpy().flatten()
            pred_mask_np = pred_labels.cpu().numpy().flatten()

            all_true_masks_flat.extend(true_mask_np)
            all_pred_masks_flat.extend(pred_mask_np)

            # Сохраняем несколько примеров для визуализации
            if visualized_count < num_samples_to_visualize:                 
                # Для получения оригинального изображения, нам нужно обратиться к исходному датасету
                # val_dataset[i] возвращает уже обработанные данные, поэтому нужно
                # получить оригинальный путь к изображению из val_dataset.dataset_pairs[i]
                original_image_path = val_dataset.dataset_pairs[i]["image"]
                original_image = cv2.imread(original_image_path)
                original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB) # Переводим в RGB

                sample_images.append(original_image)
                sample_true_masks.append(labels.cpu().squeeze().numpy())
                sample_pred_masks.append(pred_labels.cpu().squeeze().numpy())
                visualized_count += 1

    return np.array(all_true_masks_flat), np.array(all_pred_masks_flat), sample_images, sample_true_masks, sample_pred_masks

3. Функции для анализа метрик (Матрица ошибок и метрики по классам)


  Этот блок кода будет использовать собранные маски для вычисления и отображения матрицы ошибок, а также для вывода
  детальных метрик по каждому классу.

In [ ]:
def analyze_metrics(all_true_masks_flat, all_pred_masks_flat, id2label, ignore_index=255):
    """
    Вычисляет и отображает матрицу ошибок, а также метрики Precision, Recall, F1-score и IoU
    для каждого класса.

    Параметры:
        all_true_masks_flat: Одномерный массив всех истинных пиксельных меток.
        all_pred_masks_flat: Одномерный массив всех предсказанных пиксельных меток.
        id2label: Словарь, сопоставляющий ID класса его названию.
        ignore_index: Индекс класса, который следует игнорировать при расчетах.
    """
    # Фильтруем пиксели, которые должны быть проигнорированы
    valid_pixels_mask = (all_true_masks_flat != ignore_index)
    true_labels_filtered = all_true_masks_flat[valid_pixels_mask]
    pred_labels_filtered = all_pred_masks_flat[valid_pixels_mask]

    # Определяем активные классы (исключая игнорируемый)
    active_class_ids = sorted([k for k in id2label.keys() if k != ignore_index])
    active_class_names = [id2label[k] for k in active_class_ids]
    num_active_classes = len(active_class_ids)

    print("\n--- Матрица ошибок ---")
    cm = confusion_matrix(true_labels_filtered, pred_labels_filtered, labels=active_class_ids)

    # Нормализуем матрицу ошибок для отображения долей
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    cm_normalized = np.nan_to_num(cm_normalized) # Заменяем NaN на 0, если есть классы без истинных пикселей

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=active_class_names, yticklabels=active_class_names)
    plt.title("Нормализованная матрица ошибок (Confusion Matrix)")
    plt.xlabel("Предсказанный класс")
    plt.ylabel("Истинный класс")
    plt.show()

    print("\n--- Отчет по классификации (Precision, Recall, F1-score) ---")
    print(classification_report(true_labels_filtered, pred_labels_filtered,
                                labels=active_class_ids, target_names=active_class_names, zero_division=0))

    print("\n--- IoU по классам (Intersection over Union) ---")
    # Вычисляем IoU из матрицы ошибок
    iou_per_class = []
    for i in range(num_active_classes):
        true_positives = cm[i, i]
        false_positives = np.sum(cm[:, i]) - true_positives
        false_negatives = np.sum(cm[i, :]) - true_positives

        denominator = true_positives + false_positives + false_negatives
        iou = true_positives / denominator if denominator > 0 else 0
        iou_per_class.append(iou)
        print(f"Класс '{active_class_names[i]}': {iou:.4f}")

  4. Функция для визуализации предсказаний


Визуально оценим качество сегментации, отображая оригинальное изображение, истинную маску и предсказанную маску.

In [ ]:
def visualize_predictions(sample_images, sample_true_masks, sample_pred_masks, id2label, cvat_colors_rgb):
    """
    Визуализирует оригинальные изображения, истинные маски и предсказанные маски.

    Параметры:
        sample_images: Список оригинальных изображений (numpy array, RGB).
        sample_true_masks: Список истинных масок (numpy array, индексы классов).
        sample_pred_masks: Список предсказанных масок (numpy array, индексы классов).
        id2label: Словарь, сопоставляющий ID класса его названию.
        cvat_colors_rgb: Словарь, сопоставляющий название класса его RGB цвету.
    """
    def get_colored_mask(mask, id2label, cvat_colors_rgb):
        """Преобразует индексную маску в цветную RGB маску."""
        h, w = mask.shape
        colored_mask = np.zeros((h, w, 3), dtype=np.uint8)
        for class_id, class_name in id2label.items():
            if class_id == 255: # Игнорируем класс, если он не должен быть визуализирован
                continue
            color = cvat_colors_rgb.get(class_name, [0, 0, 0]) # Цвет по умолчанию черный
            colored_mask[mask == class_id] = color
        return colored_mask

    print("\n--- Визуализация предсказаний ---")
    for i in range(len(sample_images)):
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        # Оригинальное изображение
        axes[0].imshow(sample_images[i])
        axes[0].set_title("Оригинальное изображение")
        axes[0].axis("off")

        # Истинная маска
        true_colored_mask = get_colored_mask(sample_true_masks[i], id2label, cvat_colors_rgb)
        axes[1].imshow(true_colored_mask)
        axes[1].set_title("Истинная маска")
        axes[1].axis("off")

        # Предсказанная маска
        pred_colored_mask = get_colored_mask(sample_pred_masks[i], id2label, cvat_colors_rgb)
        axes[2].imshow(pred_colored_mask)
        axes[2].set_title("Предсказанная маска")
        axes[2].axis("off")

        plt.tight_layout()
        plt.show()

In [ ]:
# 1. Собираем предсказания
all_true_masks_flat, all_pred_masks_flat, sample_images, sample_true_masks, sample_pred_masks = \
    collect_predictions(model, val_dataset, device, processor, num_samples_to_visualize=5)

# 2. Анализируем метрики
analyze_metrics(all_true_masks_flat, all_pred_masks_flat, id2label, ignore_index=255)

# 3. Визуализируем предсказания
visualize_predictions(sample_images, sample_true_masks, sample_pred_masks, id2label, cvat_colors_rgb)